# Candle Notebook: Image Store + Egui Window Demo
This consolidated notebook shows how to: 
1. Configure and use the image store (relative directory `images_store`).
2. Generate and display tensors as images (grayscale and RGB).
3. Use the pseudo *egui window* HTML helpers to lay out panels.
4. Persist images to disk while also showing inline previews.

Prereqs: The helper crate `candle_notebooks` in `0aEXPLORATION/candle_notebooks` must be available to the evcxr kernel (usually by `:dep` path or installing the crate).

In [ ]:
// Link the local helper crate (adjust path if notebook moved).
:dep candle = { path = "../../candle-core" }
:dep candle-nn = { path = "../../candle-nn" }
:dep candle-notebooks = { path = "../candle_notebooks" }
// (Optional) add image / display deps if helpers rely on them.

In [ ]:
use candle::{Device, Tensor, DType};
use candle_notebooks as nb;
// Set the relative directory where generated images will be stored.
nb::set_image_store_rel_dir("images_store").unwrap();
println!("Image store directory: images_store");

## 1. Generate Sample Tensors
We'll create: 
- A simple grayscale ramp 
- A small RGB gradient image

In [ ]:
// Grayscale 16x16 ramp (values 0..255).
let w = 16; let h = 16;
let data: Vec<u8> = (0..(w*h)).map(|i| (i as f32 * 255.0 / (w*h-1) as f32) as u8).collect();
let ramp = Tensor::from_vec(data.clone(), (h, w), &Device::Cpu).unwrap();
nb::show_tensor_gray(&ramp, "ramp_gray").unwrap();
// Persist explicitly (helper may already do so depending on implementation).
nb::save_tensor_gray(&ramp, "ramp_gray_saved.png").unwrap();

In [ ]:
// RGB gradient 32x32: R varies with x, G with y, B = average.
let w = 32; let h = 32;
let mut rgb: Vec<u8> = Vec::with_capacity((w*h*3) as usize);
for y in 0..h {
  for x in 0..w {
    let r = (x as f32 * 255.0 / (w-1) as f32) as u8;
    let g = (y as f32 * 255.0 / (h-1) as f32) as u8;
    let b = (((r as u16 + g as u16)/2) as u8);
    rgb.push(r); rgb.push(g); rgb.push(b);
  }
}
let img = Tensor::from_vec(rgb, (h, w, 3), &Device::Cpu).unwrap();
nb::show_tensor_rgb(&img, "rgb_gradient").unwrap();
nb::save_tensor_rgb(&img, "rgb_gradient_saved.png").unwrap();

## 2. Compose an Egui-style Window
Use the pseudo-egui helpers to layout content blocks (HTML output).

In [ ]:
use nb::egui_window;
let panel_left = egui_window::panel("Left Panel", "<p>Some controls / text here.</p>");
let panel_right = egui_window::panel("Right Panel", "<p>Stats / logs can go here.</p>");
let layout = egui_window::two_columns(&panel_left, &panel_right, 50);
let window_html = egui_window::window("Demo Window", &layout);
nb::show_html(&window_html).unwrap();

## 3. Embed Saved Images in the Egui-style Window
We can reference the stored image filenames and build an HTML snippet to display them side-by-side.

In [ ]:
let img_html = format!("<div style='display:flex;gap:12px;'>\n  <figure><img src='images_store/ramp_gray_saved.png' width='128'/><figcaption>Gray Ramp</figcaption></figure>\n  <figure><img src='images_store/rgb_gradient_saved.png' width='128'/><figcaption>RGB Gradient</figcaption></figure>\n</div>");
let gallery_window = egui_window::window("Image Gallery", &img_html);
nb::show_html(&gallery_window).unwrap();

## 4. Notes
- The helpers auto-handle base64 inline display; explicit saves ensure persistence.
- Adjust tensor sizes or generation logic for richer demos.
- The pseudo-egui HTML is lightweight; you can style further with inline CSS if desired.